# Signed Distance Field

Mapping sampling points of UV space to nearest points on a curve.

Assuming the curve is almost flat with some little bumps/dents. Dropping $z$ for all calculations.

- $p$ — sample point in UV space
- $p^* = S(t^*)$ — nearest point on curve

$$
p|_{S(t)} \to p^* = S(t^*), t^* = \argmin_t{||p - S(t)||^2}
$$

### General solution

- $S(t)$ — general parametric curve
- $S'(t)$ ­— first derivative, velocity/flow

$$
(p - S(t))·S'(t) = 0
\tag{perpendicularity}
$$

Solving for $t$ gives points with locally minimum distance.
Globally minimum should be choosen somehow by additional computations.

### Distance

- $v_p = p - p^*$ — vector from a projection point to the sample

$$D^2 = ||v_p||^2 = v_p·v_p$$

Doesn't make sense to squaeroot because it can be used as is.

### Orientation

- $v = p - p^*$ — vector from a projection point to the sample
- $f = S'(t^*)$ ­— tangential flow at the projection point
- $z = (v × f)_z = v_x f_y - v_y f_x$ — z-component of cross product of 2D vectors into 3D

$$
\begin{cases}
z > 0 \implies \text{at right-hand side} \\
z < 0 \implies \text{at left-hand side} \\
z = 0 \implies \text{exact on the curve, should not happen}\\
\end{cases}
$$

Numeric value $z = |v||f|\sin\phi$


In [ ]:
from typing import NamedTuple, Self, Any
from collections.abc import Callable

import numpy as np
from numpy.typing import NDArray
from ipywidgets import widgets as wg
from plotly import graph_objects as go
from plotly import colors

Vec = np.ndarray[tuple[int], np.dtype[np.floating]]  # 2D or 3D
vec = lambda *args: np.array(args, dtype=np.float32)
nparr = lambda *args: np.array(args)
nparrg = lambda iter: np.array(tuple(iter))


dot = lambda v1, v2: float(v1 @ v2)

In [ ]:
def powers(d: int, t: float) -> NDArray:
    return np.power(t, np.arange(d + 1))


def powersnp(d: int, t: NDArray) -> NDArray:
    return np.power(t[..., None], np.arange(d + 1))

In [ ]:
# TODO: replace with efficient cadano/trigonometric
def npsolve3(a3: float, a2: float, a1: float, a0: float) -> set[float]:
    """Solving cubic equation by numpy"""
    roots = np.roots([a3, a2, a1, a0])
    return set(roots[roots.imag == 0].real)

In [ ]:
JIGG = nparr(0.5, 0.5)


def random_points(n: int):
    xx = np.linspace(1.0, 0.0, n)
    yy = np.linspace(0.0, 1.0, n)

    points = np.stack((xx, yy)).T
    points += (np.random.random((n, 2)) - 0.5) * 2 * JIGG
    points = np.clip(points, 0.0, 1.0)
    return points

In [ ]:
class Projection(NamedTuple):
    """Projection of a single sample point onto curve"""

    smp: Vec
    t: float
    pnt: Vec
    dsq: float
    sgn: int

    @property
    def sdist(self):
        return self.sgn * np.sqrt(self.dsq)

In [ ]:
def disquance(p0: Vec, p1: Vec):
    v = p1 - p0
    return dot(v, v)


def disquances(p0: Vec, pp: NDArray):
    vv = pp - p0
    return (vv * vv).sum(axis=1)


def orientation(forw: Vec, v: Vec):
    return np.sign(v[0] * forw[1] - v[1] * forw[0])

#### Mega widget


In [ ]:
from traitlets import Instance, Integer, observe


class SDFView(wg.Box):
    params = Instance(object)
    res_curve = Integer(default_value=5)
    res_field = Integer(default_value=64)

    plotting_fn: Callable
    projecting_fn: Callable
    picking_fn: Callable

    figure: go.FigureWidget
    lattice: NDArray

    def __init__(self, plotting_fn, projecting_fn, picking_fn, **kwargs):
        self.plotting_fn = plotting_fn
        self.projecting_fn = projecting_fn
        self.picking_fn = picking_fn

        self.figure = go.FigureWidget(
            layout=go.Layout(
                autosize=False,
                height=720,
                margin=dict(t=16, b=16, l=16, r=16),
                xaxis=dict(scaleanchor="y"),
                yaxis=dict(range=(0, 1)),
            ),
            data=[
                go.Heatmap(
                    name="field",
                    x=[0.5],
                    y=[0.5],
                    z=[0.0],
                    zmid=0.0,
                    colorscale=colors.diverging.RdBu_r,
                    showscale=False,
                    showlegend=True,
                    hovertemplate="xy: %{x:.03f} %{y:.03f}<br>dist: %{z:.03f}",
                ),
                go.Scatter(
                    name="curve",
                    x=[0, 0],
                    y=[0, 0],
                    mode="lines",
                    hovertemplate="t: %{customdata:.03f}<br>xy: %{x:.03f} %{y:.03f}",
                    line=dict(width=4, color="black"),
                    showlegend=True,
                ),
                go.Scatter(
                    name="pick",
                    x=[],
                    y=[],
                    text=[],
                    mode="markers+text",
                    marker=dict(color="purple"),
                    textposition="top center",
                    hoverinfo="skip",
                    showlegend=False,
                ),
            ],
        )

        super().__init__(children=[self.figure], **kwargs)

        self.figure.data[0].on_click(self.on_pick)

    @observe("params", "res_curve")
    def update_curve(self, _):
        if self.params is None:
            return

        tspace = np.linspace(0.0, 1.0, self.res_curve)
        curve = self.plotting_fn(self.params, tspace)

        self.figure.update_traces(
            dict(
                x=curve[:, 0],
                y=curve[:, 1],
                customdata=tspace,
            ),
            overwrite=True,
            selector={"name": "curve"},
        )

    def update_lattice(self):
        res = self.res_field
        r = 0.5 / res  # pixel half-size
        cspace = np.linspace(r, 1.0 - r, res)
        X, Y = np.meshgrid(cspace, cspace)
        self.lattice = np.stack((X, Y), axis=-1).reshape(res * res, 2)

    @observe("params", "res_field")
    def update_field(self, _):
        if self.params is None:
            return

        self.update_lattice()

        projs = [self.projecting_fn(self.params, sample) for sample in self.lattice]
        projs = [prj for prj in projs if prj is not None]
        coords = nparrg(p.smp for p in projs)
        projfield = nparrg(p.t for p in projs)
        distfield = nparrg(p.sdist for p in projs)

        self.figure.update_traces(
            dict(
                x=coords[:, 0],
                y=coords[:, 1],
                z=distfield,
                customdata=projfield,
            ),
            overwrite=True,
            selector={"name": "field"},
        )

    @observe("params")
    def update_params(self, _):
        self.reset_pick()

    def on_pick(self, trace, points, mouse):
        if len(points.point_inds) == 0:
            return

        assert len(points.point_inds) == 1

        self.update_pick(nparr(points.xs[0], points.ys[0]))

    def reset_pick(self):
        self.figure.update_traces(
            dict(
                x=[],
                y=[],
                text=[],
            ),
            {"name": "pick"},
        )

    def update_pick(self, sample: Vec):
        projs = self.picking_fn(self.params, sample)

        if len(projs):
            markers = [j.pnt for j in projs]
            labels = [f"t={j.t:.03f}" for j in projs]
        else:
            markers = []
            labels = []

        markers.append(sample)
        labels.append(f"xy: {sample[0]:.03f} {sample[1]:.03f}")

        markers = np.array(markers)

        self.figure.update_traces(
            dict(
                x=markers[:, 0],
                y=markers[:, 1],
                text=labels,
            ),
            {"name": "pick"},
        )

# Linear

Dull linear case for testing


- $c_0$ — center
- $v_1$ — direction

$$
S(t') - c_0=
\big[1, t' \big]
\begin{bmatrix}
0 \\
v_1 \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t' \big]
\begin{bmatrix}
v_1 \\
0 \\
\end{bmatrix}
$$

### Projection

- $p_s = p - c_0$ — sample point relative to center

Solution:

$t^* = p_{proj} = \frac{|v_1|p_{proj}}{|v_1|^2} = \frac{p_s · v_1}{v_1·v_1}$


In [ ]:
class LineParams(NamedTuple):
    c0: Vec
    v1: Vec


def line_points(params: LineParams, tspace: NDArray):
    return (tspace[:, None] - 0.5) * params.v1 + params.c0


def line_flow(params: LineParams, t: float):
    return params.v1

In [ ]:
def line_projection(params: LineParams, sample: Vec) -> Projection | None:
    c0, v1 = params
    ps = sample - c0

    t = dot(ps, v1) / dot(v1, v1)

    if t < -0.5 or t > 0.5:
        return None

    pnt = c0 + v1 * t
    vp = sample - pnt
    dsq = dot(vp, vp)
    sgn = np.sign(vp[0] * v1[1] - vp[1] * v1[0])

    return Projection(sample, t, pnt, dsq, sgn)


def line_projections(params: LineParams, sample: Vec) -> tuple[Projection, ...]:
    proj = line_projection(params, sample)
    if proj is None:
        return ()
    else:
        return (proj,)

In [ ]:
view = SDFView(plotting_fn=line_points, projecting_fn=line_projection, picking_fn=line_projections)

display(view)

In [ ]:
view.params = LineParams(nparr(0.5, 0.5), np.random.random(2) - 0.5)

In [ ]:
view.res_field = 32

# Quadratic Bezier

Bezier segment over $c_1, c_2, c_3$ localized around its apex with $t=0.0$.


- $v_1 = c_1 - c_2$
- $v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axial vector
- $v_t = v_3 - v_1$ — parabolic tangential vector
- $c_a = c_2 + .25 v_a = S(0)$ — apex point

$$
S(t') - c_a=
\big[1, t', t'^2 \big]
\begin{bmatrix}
0 \\
v_t \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t', t'^2 \big]
\begin{bmatrix}
v_t \\
2 v_a \\
0 \\
\end{bmatrix}
$$


### Distance

- $p_a = p - c_a$ — point relative to apex
- $v_p(t)|p = p - S(t)$ ­— vector from curve to sample point
- $v_p^2 = v_p·v_p = |v_p|^2$ — squared distance from curve to point

Equation for local minima/maxima:

$$
\frac{dv_p^2}{dt} =
[1, t, t^2, t^3]
\begin{pmatrix}
-p_a v_t \\
-2 p_a v_a + v_t^2 \\
3 v_a v_t \\
2 v_a^2 \\
\end{pmatrix}
= 0
$$

There are 1 up to 3 bending points. Need to select the best.


### Visibility

$$
\begin{cases}
(p - c_2) · v_1 > v_1 · v_1 \implies t_{prj} < -0.5
\\
(p - c_2) · v_3 > v_3 · v_3 \implies t_{prj} > +0.5
\end{cases}
$$

Indicates that a sample projection guarantied to lie behind horizons of segment


In [ ]:
class SegmParams(NamedTuple):
    """Params of a bezier segment on points c_1 c_2 c_3"""

    c2: Vec  # zenith
    v1: Vec  # c2 > c1
    v3: Vec  # c2 > c3

    ca: Vec  # apex
    vt: Vec  # parabolic tangent
    va: Vec  # parabolix axis

    @classmethod
    def Init(cls, c1: Vec, c2: Vec, c3: Vec) -> Self:
        v1 = c1 - c2
        v3 = c3 - c2
        vt = v3 - v1
        va = v3 + v1
        ca = c2 + 0.25 * va
        return cls(c2, v1, v3, ca, vt, va)

    @classmethod
    def Rev(cls, ca: Vec, vt: Vec, va: Vec):
        c2 = ca - 0.25 * va
        v3 = (va + vt) * 0.5
        v1 = (va - vt) * 0.5
        return cls(c2, v1, v3, ca, vt, va)

    @property
    def M(self):
        return nparr(np.zeros(self.c2.shape), self.vt, self.va)

    @property
    def Mdt(self):
        return nparr(self.vt, 2 * self.va, np.zeros(self.c2.shape))


def segm_point(params: SegmParams, t: float):
    return powers(2, t - 0.5) @ params.M + params.ca


def segm_curve(params: SegmParams, tspace: NDArray):
    return powersnp(2, tspace - 0.5) @ params.M + params.ca


def segm_tang(params: SegmParams, t: float):
    return powers(2, t - 0.5) @ params.Mdt


def segm_flow(params: SegmParams, tspace: NDArray):
    return powersnp(2, tspace - 0.5) @ params.Mdt

In [ ]:
def segm_visible(params: SegmParams, sample: Vec):
    """Check if the sample is visible within segment horizons"""
    c2, v1, v3, _, _, _ = params
    pc = sample - c2
    pc1 = dot(pc, v1)
    pc3 = dot(pc, v3)
    return pc1 <= dot(v1, v1) or pc3 <= dot(v3, v3)

In [ ]:
def segm_equation(params: SegmParams, sample: Vec) -> NDArray:
    """Solve the equation of perpendicularity = square distance local extrema
    a0 + a1 t + a2 t^2 + a3 t^3
    """
    _, _, _, ca, vt, va = params
    p = ca - sample  # -p_a
    dot_aa = dot(va, va)
    dot_at = dot(va, vt)
    dot_tt = dot(vt, vt)
    dot_pa = dot(p, va)
    dot_pt = dot(p, vt)

    a0 = dot_pt
    a1 = 2 * dot_pa + dot_tt
    a2 = 3 * dot_at
    a3 = 2 * dot_aa

    roots = np.roots([a3, a2, a1, a0])
    roots_real = roots[roots.imag == 0].real
    roots_uniq = np.fromiter(set(roots_real), dtype=float)

    return roots_uniq


def segm_projections_all(params: SegmParams, sample: Vec):
    tt = segm_equation(params, sample) + 0.5
    pnts = segm_curve(params, tt)
    dsqs = disquances(sample, pnts)
    return [
        Projection(
            sample,
            t,
            pnt,
            dsq,
            orientation(sample - pnt, segm_tang(params, t)),
        )
        for t, pnt, dsq in zip(tt, pnts, dsqs)
    ]


def segm_projection_best(params: SegmParams, sample: Vec):
    tt = segm_equation(params, sample) + 0.5
    pnts = segm_curve(params, tt)
    dsqs = disquances(sample, pnts)

    best = np.argmin(dsqs)
    t = tt[best]
    pnt = pnts[best]
    dsq = dsqs[best]

    return Projection(
        sample,
        t,
        pnt,
        dsq,
        orientation(segm_tang(params, t), sample - pnt),
    )

In [ ]:
view = SDFView(plotting_fn=segm_curve, projecting_fn=segm_projection_best, picking_fn=segm_projections_all, res_curve=32)

display(view)

In [ ]:
controls = random_points(3)
view.params = SegmParams.Init(*controls)